## Imports

In [67]:
import sys
import os
import torch
import torch.utils.data as data
import torch.nn as nn

sys.path.append(os.path.abspath('../../'))

import neuro_fuzzy_toolbox as nft

In [68]:
import numpy as np

In [69]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    classification_report,
    multilabel_confusion_matrix
)

from sklearn import preprocessing

In [70]:
from ucimlrepo import fetch_ucirepo

# Iris

Se observa que con regresiones logisticas los resultados son suficientemente competitivos (https://archive.ics.uci.edu/dataset/53/iris), lo cual explica porque con una o dos reglas basta para obtener un buen resultado.

## Data

In [71]:
iris = fetch_ucirepo(id=53)

X = iris.data.features
y = iris.data.targets

In [72]:
iris.variables

,name,role,type,demographic,description,units,missing_values
0,sepal length,Feature,Continuous,None,None,cm,no
1,sepal width,Feature,Continuous,None,None,cm,no
2,petal length,Feature,Continuous,None,None,cm,no
3,petal width,Feature,Continuous,None,None,cm,no
4,class,Target,Categorical,None,"class of iris plant: Iris Setosa, Iris Versico...",None,no


In [73]:
X

,sepal length,sepal width,petal length,petal width
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2
...,...,...,...,...
145,6.7,3.0,5.2,2.3
146,6.3,2.5,5.0,1.9
147,6.5,3.0,5.2,2.0
148,6.2,3.4,5.4,2.3


In [74]:
y

,class
0,Iris-setosa
1,Iris-setosa
2,Iris-setosa
3,Iris-setosa
4,Iris-setosa
...,...
145,Iris-virginica
146,Iris-virginica
147,Iris-virginica
148,Iris-virginica


In [75]:
le = preprocessing.LabelEncoder()
y.loc[:, 'class'] = le.fit_transform(y['class'])

In [76]:
y = y.astype('int64')
y

,class
0,0
1,0
2,0
3,0
4,0
...,...
145,2
146,2
147,2
148,2


In [77]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.2, stratify=y_train)

In [78]:
unique_classes, counts = np.unique(y_train.values, return_counts=True)
print("Training Data")
print(unique_classes)
print(counts)

Training Data
[0 1 2]
[28 28 28]


In [79]:
unique_classes, counts = np.unique(y_val.values, return_counts=True)
print("Validation Data")
print(unique_classes)
print(counts)

Validation Data
[0 1 2]
[7 7 7]


In [80]:
unique_classes, counts = np.unique(y_test.values, return_counts=True)
print("Testing Data")
print(unique_classes)
print(counts)

Testing Data
[0 1 2]
[15 15 15]


In [81]:
scaler = MinMaxScaler(feature_range=(-1, 1))

x_train = scaler.fit_transform(x_train)

x_val = scaler.transform(x_val)
x_test = scaler.transform(x_test)

In [82]:
y_train.values.dtype

dtype('int64')

In [83]:
x_train = torch.from_numpy(x_train)
x_val = torch.from_numpy(x_val)
x_test = torch.from_numpy(x_test)

y_train = torch.from_numpy(y_train.values).squeeze()
y_val = torch.from_numpy(y_val.values).squeeze()
y_test = torch.from_numpy(y_test.values).squeeze()

## Model & Algorithms

### DataLoaders

In [84]:
train_loader = data.DataLoader(
    data.TensorDataset(x_train, y_train), 
    batch_size = 8, 
    shuffle = True)
x_train = train_loader.dataset.tensors[0]
y_train = train_loader.dataset.tensors[1]

In [85]:
val_loader = data.DataLoader(
    data.TensorDataset(x_val, y_val), 
    batch_size = 8, 
    shuffle = True)
x_val = val_loader.dataset.tensors[0]
y_val = val_loader.dataset.tensors[1]

### ANFIS

In [86]:
features = iris.variables["name"][:4].tolist()
features

['sepal length', 'sepal width', 'petal length', 'petal width']

In [87]:
model = nft.rule_reduced_ANFIS(
    input_size = x_train.shape[1],
    num_mfs = 1, # 2 - 3
    outputs = 3,
    default_rule=True,
    output_type='softmax',
    features=features,
    dtype=torch.float64
)

#model.init_premises(x_train)

In [88]:
with torch.no_grad():
    pred = model(x_train)

nn.functional.cross_entropy(pred, y_train)

tensor(1.0756, dtype=torch.float64)

In [89]:
model.init_consequents(x_train, y_train, driver="gelsd", ridge_lambda=1.0)
"""
with torch.no_grad():
    pred = model(x_train)

nn.functional.cross_entropy(pred, y_train)
"""

'\nwith torch.no_grad():\n    pred = model(x_train)\n\nnn.functional.cross_entropy(pred, y_train)\n'

### Learning Algorithm

In [90]:
#loss_fn = nn.functional.binary_cross_entropy
loss_fn = nn.functional.cross_entropy
epochs = 200
#optimizer = torch.optim.Adam
#params = {'lr': 0.001}
optimizer = torch.optim.AdamW
params = {'lr': 0.001, 'weight_decay': 0.01}
#optimizer = torch.optim.SGD
#params = {'lr': 0.001, 'momentum': 0.9}

early_stopping = nft.EarlyStopping(
    patience=30, 
    #delta=0.0001
)

In [91]:
trainer = nft.Basic_optimizer_training_algorithm(
    epochs=epochs,
    loss_function=loss_fn,
    optimizer=optimizer,
    optimizer_params=params,
    early_stopping=early_stopping
)

In [92]:
"""
trainer(model, train_loader, val_loader, verbose=True)

pred = model.predict(x_test)

acc = accuracy_score(y_test, pred, )
prec = precision_score(y_test, pred, average='weighted', zero_division=0)
recall = recall_score(y_test, pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, pred, average='weighted', zero_division=0)
conf_matrix = confusion_matrix(y_test, pred)
class_rep = classification_report(y_test, pred)

print("\n ------ Evaluation ------")
print("\n")
print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", recall)
print("f1 score:", f1, "\n")

print("Confusion Matrix:")
print(conf_matrix, "\n")

print("Classification Report:")
print(class_rep)
"""

'\ntrainer(model, train_loader, val_loader, verbose=True)\n\npred = model.predict(x_test)\n\nacc = accuracy_score(y_test, pred, )\nprec = precision_score(y_test, pred, average=\'weighted\', zero_division=0)\nrecall = recall_score(y_test, pred, average=\'weighted\', zero_division=0)\nf1 = f1_score(y_test, pred, average=\'weighted\', zero_division=0)\nconf_matrix = confusion_matrix(y_test, pred)\nclass_rep = classification_report(y_test, pred)\n\nprint("\n ------ Evaluation ------")\nprint("\n")\nprint("Accuracy:", acc)\nprint("Precision:", prec)\nprint("Recall:", recall)\nprint("f1 score:", f1, "\n")\n\nprint("Confusion Matrix:")\nprint(conf_matrix, "\n")\n\nprint("Classification Report:")\nprint(class_rep)\n'

### SONFIS

In [93]:
Ngrow = 40
dGrow = 0.9
Nsplit = 60
eSplit = 0.2
Nvanish = 10
lVanish = 3

max_iterations = 100

anfis_trainer = trainer

sonfis_early_stopping = nft.EarlyStopping(patience=20)
last_training_iteration = False

lse_for_new_consequents = True
lse_for_new_consequents_lambda = 1e-2

In [94]:
sonfis = nft.SONFIS(
    Ngrow=Ngrow,
    dGrow=dGrow,
    Nsplit=Nsplit,
    eSplit=eSplit,
    Nvanish=Nvanish,
    lVanish=lVanish,
    max_iterations=max_iterations,
    ANFIStrainer=anfis_trainer,
    early_stopping=sonfis_early_stopping,
    lse_for_new_consequents=lse_for_new_consequents,
    #lse_for_new_consequents_lambda=lse_for_new_consequents_lambda,
    last_training_iteration=last_training_iteration
)

In [95]:
np.int64(4)

np.int64(4)

## Training

In [96]:
%time sonfis(model, train_loader, val_loader, verbose=True)

ITERATION:   0/100



STARTING STATE:
      premises                                                                                                                     output 1 consequents                                                output 2 consequents                                                output 3 consequents                                              
  sepal length                     sepal width                     petal length                    petal width                             sepal length sepal width petal length petal width                   sepal length sepal width petal length petal width                   sepal length sepal width petal length petal width         
             a         b         c           a         b         c            a         b        c           a         b         c                   c0          c1           c2          c3        c4                   c0          c1           c2          c3        c4                   c0          c1           c2   

## Evaluation

### Test data

In [97]:
pred = model.predict(x_test)

acc = accuracy_score(y_test, pred)
prec = precision_score(y_test, pred, average='weighted', zero_division=0)
recall = recall_score(y_test, pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, pred, average='weighted', zero_division=0)
conf_matrix = confusion_matrix(y_test, pred)
#mul_label_conf_matrix = multilabel_confusion_matrix(y_test, pred)
class_rep = classification_report(y_test, pred)

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", recall)
print("f1 score:", f1, "\n")

print("Confusion Matrix:")
print(conf_matrix, "\n")

#print("Multilabel Confusion Matrix:")
#print(mul_label_conf_matrix, "\n")

print("Classification Report:")
print(class_rep)

Accuracy: 0.9555555555555556
Precision: 0.9555555555555556
Recall: 0.9555555555555556
f1 score: 0.9555555555555556 

Confusion Matrix:
[[15  0  0]
 [ 0 14  1]
 [ 0  1 14]] 

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        15
           1       0.93      0.93      0.93        15
           2       0.93      0.93      0.93        15

    accuracy                           0.96        45
   macro avg       0.96      0.96      0.96        45
weighted avg       0.96      0.96      0.96        45



### Training data

In [98]:
pred = model.predict(x_train)

acc = accuracy_score(y_train, pred)
prec = precision_score(y_train, pred, average='weighted', zero_division=0)
recall = recall_score(y_train, pred, average='weighted', zero_division=0)
f1 = f1_score(y_train, pred, average='weighted', zero_division=0)
conf_matrix = confusion_matrix(y_train, pred)
class_rep = classification_report(y_train, pred)

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", recall)
print("f1 score:", f1, "\n")

print("Confusion Matrix:")
print(conf_matrix, "\n")

print("Classification Report:")
print(class_rep)

Accuracy: 0.9880952380952381
Precision: 0.9885057471264369
Recall: 0.9880952380952381
f1 score: 0.9880914407230196 

Confusion Matrix:
[[28  0  0]
 [ 0 28  0]
 [ 0  1 27]] 

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        28
           1       0.97      1.00      0.98        28
           2       1.00      0.96      0.98        28

    accuracy                           0.99        84
   macro avg       0.99      0.99      0.99        84
weighted avg       0.99      0.99      0.99        84



In [99]:
print(model.get_rules_structure().to_string())

           premises                                                                                                                      output 1 consequents                                                output 2 consequents                                                output 3 consequents                                               
       sepal length                     sepal width                     petal length                     petal width                             sepal length sepal width petal length petal width                   sepal length sepal width petal length petal width                   sepal length sepal width petal length petal width          
                  a         b         c           a         b         c            a         b         c           a         b         c                   c0          c1           c2          c3        c4                   c0          c1           c2          c3        c4                   c0          c1           c2